# Stage 1: Environment Setup
Run this cell first. It uses the `%pip` magic command to ensure pandas and visualization libraries are installed into the EXACT Python kernel this notebook is using.

In [ ]:
%pip install pandas matplotlib seaborn

# Stage 2: Invoice Data Analysis & Visualization
Now that the heavy GPU lifting is done, we can analyze the pure text data extremely quickly on our local CPU.

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")

# 1. Load the completed JSONL data
file_path = 'gstinfo.jsonl'

records = []
with open(file_path, 'r') as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)

print(f"Successfully loaded {len(df)} extracted invoices.")
display(df.head())

In [ ]:
# 2. Analyze Extraction Success Rates
core_fields = ['gst_number', 'vendor_name', 'vendor_address']

success_rates = {}
for field in core_fields:
    if field in df.columns:
        # Check if it's not strictly empty or the literal string "null"
        valid_mask = df[field].notna() & (df[field].astype(str).str.lower() != "null") & (df[field] != "")
        success_rates[field] = valid_mask.mean() * 100

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=list(success_rates.keys()), y=list(success_rates.values()), palette="viridis")
plt.title('VLM Extraction Success Rate per Field', fontsize=16)
plt.ylabel('Success Percentage (%)', fontsize=12)
plt.ylim(0, 100)

for i, v in enumerate(success_rates.values()):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.show()

In [ ]:
# 3. Data Cleaning & Top Vendors Visualization
valid_gst_mask = df['gst_number'].notna() & (df['gst_number'].astype(str).str.lower() != "null") & (df['gst_number'] != "")
clean_df = df[valid_gst_mask].copy()

print(f"We have {len(clean_df)} invoices with successfully extracted GST numbers.\n")

if 'vendor_name' in clean_df.columns:
    # Standardize vendor names (uppercase, strip whitespace)
    clean_df['vendor_name_clean'] = clean_df['vendor_name'].astype(str).str.upper().str.strip()
    
    top_vendors = clean_df['vendor_name_clean'].value_counts().head(10)
    
    plt.figure(figsize=(12, 7))
    sns.barplot(y=top_vendors.index, x=top_vendors.values, palette="mako")
    plt.title('Top 10 Most Frequent Vendors by Invoice Count', fontsize=16)
    plt.xlabel('Number of Invoices', fontsize=12)
    plt.ylabel('Vendor Name', fontsize=12)
    plt.show()